### Environment and Library Workarounds
This cell imports the core libraries (like `google.cloud.storage` and others) and injects a dummy `PeftConfigLike` class into Python's builtins to bypass a crash in Python 3.12 type-hint evaluation during PEFT import.

In [ ]:
import builtins


# The Magic Hack: Create a dummy class and inject it into Python's builtins
# so the Python 3.12 type-hint evaluator finds it and stops crashing.
class DummyPeftConfig:
    pass


if not hasattr(builtins, "PeftConfigLike"):
    builtins.PeftConfigLike = DummyPeftConfig

import os
import sys
import json
import torch
import logging
from google.cloud import storage

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

### Configuration and Hyperparameters
Define training configurations, model choice (e.g., `openai/whisper-tiny`), paths to dataset manifests in GCS or locally, PEFT (LoRA) parameters, training hyperparameters (batch size, learning rate, etc.), and output directories.

In [ ]:
# Start with the smallest model to verify code execution without OOM
# MODEL_NAME = "openai/whisper-tiny"
MODEL_NAME = "openai/whisper-large-v3-turbo"  # Options: "openai/whisper-large-v3", "openai/whisper-large-v3-turbo"

# Whisper models natively expect audio sampled at 16,000 Hz
TARGET_SAMPLE_RATE = 16000

# Dataset manifest paths.
# Can be local file paths (e.g. "/path/to/manifest.jsonl") or GCS URIs (e.g. "gs://bucket/manifest.jsonl")
TRAIN_MANIFEST_PATH = ""
VAL_MANIFEST_PATH = ""

assert TRAIN_MANIFEST_PATH, "TRAIN_MANIFEST_PATH must be provided and cannot be empty."
assert VAL_MANIFEST_PATH, "VAL_MANIFEST_PATH must be provided and cannot be empty."

# PEFT configuration (highly recommended to avoid OOM on free colab GPUs)
USE_PEFT = True
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# TODO(varun): Review these parameters
# Training arguments
BATCH_SIZE = 4  # Reduce batch size (e.g. to 1 or 2) if you encounter OOM on GPU
GRADIENT_ACCUMULATION_STEPS = (
    4  # Simulates larger batch sizes without memory penalty
)
LEARNING_RATE = 1e-5
WARMUP_STEPS = 2
MAX_STEPS = 2000  # Adjust to run more training steps
EVAL_STEPS = 100
SAVE_STEPS = 100
LOGGING_STEPS = 25
MAX_EVAL_SAMPLES = (
    100  # Limit validation dataset size to save time during evaluation
)

GCP_PROJECT_ID = "<GCP-PROJECT-ID>"
ROOT_OUTPUT_DIR = "../trained_checkpoints"
EXPERIMENT_NAME = "whisper-turbo-experiment"
EXPERIMENT_DIR = f"{ROOT_OUTPUT_DIR}/{EXPERIMENT_NAME}"

### Audio and Dataset Helpers
Define functions to resolve, cache, download, and load audio files. Implement `NemoDataset`, a custom PyTorch `Dataset` for processing NeMo-formatted audio-transcription manifests and preparing them for Whisper.

In [ ]:
import os
import json
import urllib.parse
import torch
import torchaudio
import torchaudio.transforms as T
import soundfile as sf
from torch.utils.data import Dataset


def get_local_audio_path(
    audio_filepath, storage_client=None, cache_dir="/tmp/audio_cache"
):
    """Resolves audio file path. If it is a GCS path, downloads and caches it locally."""
    os.makedirs(cache_dir, exist_ok=True)
    if audio_filepath.startswith("gs://"):
        if storage_client is None:
            raise ValueError(
                "storage_client must be provided to download GCS paths"
            )

        # Parse GCS URI
        parsed = urllib.parse.urlparse(audio_filepath)
        bucket_name = parsed.netloc
        blob_name = parsed.path.lstrip("/")

        # Create a safe local file name using bucket and path to avoid collision
        safe_name = f"{bucket_name}_{blob_name.replace('/', '_')}"
        local_path = os.path.join(cache_dir, safe_name)

        if not os.path.exists(local_path):
            # print(f"Downloading {audio_filepath} to local cache...")
            bucket = storage_client.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            blob.download_to_filename(local_path)
        return local_path
    else:
        return audio_filepath


def load_audio_segment(
    filepath, offset=0.0, duration=None, target_sr=TARGET_SAMPLE_RATE
):
    """Loads a segment of an audio file, downmixes it to mono, and resamples it to target_sr."""
    try:
        # Read audio metadata first to obtain sample rate using soundfile
        info = sf.info(filepath)
        sr = info.samplerate

        # Calculate frame offset and count based on original sample rate
        frame_offset = int(offset * sr) if offset is not None else 0
        num_frames = int(duration * sr) if duration is not None else -1

        if frame_offset < 0:
            frame_offset = 0
        if num_frames < 0:
            num_frames = -1

        waveform, sample_rate = torchaudio.load(
            filepath, frame_offset=frame_offset, num_frames=num_frames
        )

        # Average to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Resample to target_sr
        if sample_rate != target_sr:
            resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sr)
            waveform = resampler(waveform)

        return waveform.squeeze(0)
    except Exception as e:
        logger.error(
            f"Error loading audio segment from {filepath} (offset: {offset}, duration: {duration}): {e}"
        )
        raise e


class NemoDataset(Dataset):
    """Custom Dataset for NeMo manifests."""

    def __init__(
        self,
        manifest_path_or_uri,
        processor,
        storage_client=None,
        cache_dir="/tmp/audio_cache",
    ):
        self.entries = []
        self.processor = processor
        self.storage_client = storage_client
        self.cache_dir = cache_dir

        # Resolve manifest path (handles GCS and local files)
        if manifest_path_or_uri.startswith("gs://"):
            if storage_client is None:
                raise ValueError(
                    "storage_client must be provided to download GCS manifest"
                )
            # Download manifest
            parsed = urllib.parse.urlparse(manifest_path_or_uri)
            bucket = storage_client.bucket(parsed.netloc)
            blob = bucket.blob(parsed.path.lstrip("/"))

            manifest_content = blob.download_as_text()
            lines = manifest_content.strip().split("\n")
        else:
            with open(manifest_path_or_uri, "r", encoding="utf-8") as f:
                lines = f.readlines()

        for line in lines:
            if line.strip():
                self.entries.append(json.loads(line))

        print(
            f"Loaded {len(self.entries)} entries from manifest: {manifest_path_or_uri}"
        )

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        audio_filepath = entry["audio_filepath"]
        text = entry["text"]
        # Manually hacking this because we are using batch manifests which currently
        # still contain the offset and duration of the original segment (before
        # clipping the audio). We will fix these manifests to relabel the 'offset'
        # and 'duration' fields to 'original_offset' and 'original_duration', and we
        # can then remove this hack.
        offset = 0.0
        duration = None

        # 1. Get local path to the audio file (handles on-demand GCS download)
        local_path = get_local_audio_path(
            audio_filepath, self.storage_client, self.cache_dir
        )

        # 2. Load the audio segment at target sample rate
        waveform = load_audio_segment(
            local_path, offset, duration, target_sr=TARGET_SAMPLE_RATE
        )

        # 3. Extract input_features
        # Note: Whisper processor handles standard padding to 30s
        # TODO: What happens when audio is > 30 seconds in this processor?
        inputs = self.processor(
            audio=waveform.numpy(), sampling_rate=TARGET_SAMPLE_RATE
        )
        input_features = inputs.input_features[0]

        # 4. Tokenize transcription
        labels = self.processor.tokenizer(text).input_ids

        return {"input_features": input_features, "labels": labels}

### Initialize Processor, Data Collator, and Evaluation Metrics
Initialize the Whisper processor (combining feature extractor and tokenizer), set up the evaluation metric (Word Error Rate or WER), and define `DataCollatorSpeechSeq2SeqWithPadding`, which dynamically pads audio input features and text labels in each batch during training.

In [ ]:
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import EvalPrediction, WhisperProcessor

print(f"Loading processor for {MODEL_NAME}...")
# Load processor (combines feature extractor and tokenizer)
processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language="english", task="transcribe"
)

# Load Word Error Rate (WER) metric
wer_metric = evaluate.load("wer")


# TODO(varun): Use our shared library metrics for this (esp
# the part about normalizing numbers etc). Need to check
# speed though, as this eval is running sequentially with the training
# and our current eval is not that fast.
def compute_metrics(pred: EvalPrediction) -> Dict[str, float]:
    """Computes Word Error Rate (WER) for the model predictions."""
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode prediction and label ids back to strings
    pred_str = processor.tokenizer.batch_decode(
        pred_ids, skip_special_tokens=True
    )
    label_str = processor.tokenizer.batch_decode(
        label_ids, skip_special_tokens=True
    )

    # Compute Word Error Rate (WER)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Data collator that dynamically pads input features and tokenized labels,
    stacking individual dataset examples into unified batches to pass downstream to the trainer.
    """

    processor: Any

    def __call__(
        self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:
        # Whisper processor has already padded the audio inputs to 30s (3000 frames)
        input_features = [
            {"input_features": feature["input_features"]}
            for feature in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        # Pad labels to max length in the current batch
        label_features = [
            {"input_ids": feature["labels"]} for feature in features
        ]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        # Replace pad token id by -100 so that CrossEntropyLoss ignores padding when computing loss
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # TODO(varun): Do we need to remove the BOS(beginning of sentence) token from the label?
        # See: https://colab.research.google.com/github/sanchit-gandhi/notebooks/blob/main/fine_tune_whisper.ipynb#scrollTo=8326221e-ec13-4731-bb4e-51e5fc1486c5&line=7&uniqifier=1

        # Set target labels
        batch["labels"] = labels
        return batch


# Initialize data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

### Dataset Loading
Initialize the GCP Storage Client and instantiate the train and validation dataset objects using the configured manifest paths. Also downsample the validation dataset if a limit (`MAX_EVAL_SAMPLES`) is configured.

In [ ]:
from torch.utils.data import Subset

# Initialize GCP Storage Client
storage_client = None
if GCP_PROJECT_ID and GCP_PROJECT_ID != "<YOUR_GCP_PROJECT_ID>":
    print(f"Initializing GCP Storage Client for project {GCP_PROJECT_ID}...")
    storage_client = storage.Client(project=GCP_PROJECT_ID)

# Initialize NemoDatasets
print("Loading training dataset...")
train_dataset = NemoDataset(
    manifest_path_or_uri=TRAIN_MANIFEST_PATH,
    processor=processor,
    storage_client=storage_client,
)

print("Loading validation dataset...")
val_dataset = NemoDataset(
    manifest_path_or_uri=VAL_MANIFEST_PATH,
    processor=processor,
    storage_client=storage_client,
)

if MAX_EVAL_SAMPLES and len(val_dataset) > MAX_EVAL_SAMPLES:
    print(
        f"Using a subset of validation dataset with {MAX_EVAL_SAMPLES} samples..."
    )
    val_dataset = Subset(val_dataset, range(MAX_EVAL_SAMPLES))

### Initialize Model and PEFT/LoRA
Load the pre-trained Whisper model. Optionally configure and apply parameter-efficient fine-tuning (PEFT/LoRA) to significantly reduce memory usage.

In [ ]:
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"Loading model {MODEL_NAME}...")
# Load Whisper model
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    # This is causig problems, experimenting.
    # torch_dtype=torch.float16,
    torch_dtype=torch.float32,
    device_map="auto",
)

# Override configuration parameters
# TODO(varun): The agent added these configs, but I didn't quite understand
# its explanation when I asked what these do, needs more background understanding
# of Whisper and how it uses some specific tokens during training.
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []


# TODO(varun): Restrict model language to english only and task to transcribe?
# See this colab for how: https://colab.research.google.com/github/sanchit-gandhi/notebooks/blob/main/fine_tune_whisper.ipynb#scrollTo=62038ba3-88ed-4fce-84db-338f50dcd04f&line=2&uniqifier=1

if USE_PEFT:
    print("Configuring and applying PEFT/LoRA...")
    # Prepare model for FP16 training with gradient checkpointing
    model.gradient_checkpointing_enable()

    # TODO(varun): Look at the whisper architecture carefully to see whether
    # these q_proj and v_proj matrices cover the audio encoding process as
    # well.
    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj",
            "v_proj",
        ],  # Whisper's multi-head attention modules
        lora_dropout=LORA_DROPOUT,
        bias="none",
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
else:
    print("PEFT disabled. Training full parameters.")

### Trainer Initialization
Set up the Hugging Face `Seq2SeqTrainingArguments` and initialize the `Seq2SeqTrainer` with the model, datasets, evaluation metrics, and custom data collator.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=EXPERIMENT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    # TODO(varun): review this
    gradient_checkpointing=False,
    # This is causing issues if set to True right now
    fp16=False,
    eval_strategy="steps",
    per_device_eval_batch_size=BATCH_SIZE,
    predict_with_generate=True,
    # TODO(varun): Review -- what about training?
    generation_max_length=225,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    logging_steps=LOGGING_STEPS,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    remove_unused_columns=False,  # crucial since our custom Dataset doesn't match the signature columns
)

# Initialize Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print("Seq2SeqTrainer initialized successfully.")

### Run Fine-Tuning
Start model fine-tuning. After training finishes, save the final model checkpoints and processor configurations to the output directory.

In [ ]:
print("Starting training...")

print("Using device: ", trainer.args.device)
trainer.train()

# Save the final fine-tuned model locally
print("Training completed. Saving fine-tuned model...")
trainer.save_model(EXPERIMENT_DIR)
processor.save_pretrained(EXPERIMENT_DIR)
print(f"Model and processor successfully saved to {EXPERIMENT_DIR}.")